# Smart Medical Analytics - Full Pipeline
Run all modules end-to-end and save trained models with joblib.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from src import classification, clustering, naive_bayes, preprocessing, regression

DATA_DIR = Path('data')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
# Regression: insurance.csv -> charges
reg_model, reg_metrics = regression.run_regression(str(DATA_DIR / 'insurance.csv'))
joblib.dump(reg_model, MODEL_DIR / 'linear_regression.joblib')
print('Linear Regression Metrics:')
reg_metrics

In [ ]:
# Logistic Regression: dataset.csv or heart.csv
log_model, log_metrics, log_extra = classification.run_logistic(
    str(DATA_DIR / 'dataset.csv'), str(DATA_DIR / 'heart.csv')
)
joblib.dump(log_model, MODEL_DIR / 'logistic_regression.joblib')
print('Logistic Regression Metrics:')
log_metrics

In [ ]:
# Naive Bayes: symptom_Description.csv
nb_model, nb_metrics, nb_extra = naive_bayes.run_naive_bayes(
    str(DATA_DIR / 'symptom_Description.csv')
)
joblib.dump(nb_model, MODEL_DIR / 'naive_bayes.joblib')
print('Naive Bayes Metrics:')
nb_metrics

In [ ]:
# Clustering and PCA: heart.csv
heart_df = preprocessing.load_csv(str(DATA_DIR / 'heart.csv'))
clust_data = clustering.prepare_clustering_data(heart_df)

k_values = list(range(2, 11))
inertias = clustering.kmeans_elbow(clust_data.features, k_values)
optimal_k = clustering.find_elbow(k_values, inertias)

kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans_model.fit_predict(clust_data.features)
joblib.dump(kmeans_model, MODEL_DIR / 'kmeans.joblib')

hier_model = AgglomerativeClustering(n_clusters=4)
hier_labels = hier_model.fit_predict(clust_data.features)
joblib.dump(hier_model, MODEL_DIR / 'hierarchical.joblib')

dbscan_model = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan_model.fit_predict(clust_data.features)
joblib.dump(dbscan_model, MODEL_DIR / 'dbscan.joblib')
noise_points = int(np.sum(dbscan_labels == -1))

pca_model = PCA(n_components=2, random_state=42)
pca_points = pca_model.fit_transform(clust_data.features)
joblib.dump(pca_model, MODEL_DIR / 'pca.joblib')

results = {
    'optimal_k': optimal_k,
    'kmeans_clusters': int(np.unique(kmeans_labels).size),
    'hier_clusters': int(np.unique(hier_labels).size),
    'dbscan_clusters': int(np.unique(dbscan_labels).size - (1 if -1 in dbscan_labels else 0)),
    'dbscan_noise_points': noise_points
}
print('Clustering Results:')
results

In [ ]:
# Verify all models are saved
saved_models = list(MODEL_DIR.glob('*.joblib'))
print(f'Saved {len(saved_models)} models:')
for model in sorted(saved_models):
    print(f'  - {model.name}')